<a href="https://colab.research.google.com/github/uiuxkelompok/utsdinda/blob/main/Google_Colab_Dinda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google_play_scraper import reviews, Sort
import csv

result, _ = reviews(
    'com.block.juggle',
    lang='id',
    country='id',
    sort=Sort.NEWEST,
    count=100,
    filter_score_with=None
)

filename = 'ulasan_google_play.csv'


with open(filename, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['userName', 'score', 'at', 'content'])
    writer.writeheader()
    for review in result:

        writer.writerow({
            'userName': review['userName'],
            'score': review['score'],
            'at': review['at'],
            'content': review['content']
        })

print(f"Berhasil menyimpan {len(result)} ulasan ke '{filename}'")

In [ ]:
import pandas as pd
!pip install transformers torch

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# Load the CSV file into a pandas DataFrame (using the updated filename from the previous step)
df_reviews = pd.read_csv(filename)

# Display the first few rows to confirm it's loaded correctly
print("Original reviews DataFrame:")
display(df_reviews.head())

In [ ]:
# Load the tokenizer and model
model_name = 'w11wo/indonesian-roberta-base-prdect-id'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Check if CUDA is available and move the model to GPU if it is
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Using device: {device}")

In [ ]:
def get_sentiment(text):
    if not isinstance(text, str): # Handle non-string inputs, e.g., NaN
        return None
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    probabilities = torch.softmax(outputs.logits, dim=1)

    # The model's labels might be 0 for negative, 1 for neutral, 2 for positive
    # or a different mapping. We use model.config.id2label for the exact mapping.
    labels = model.config.id2label
    predicted_class_id = probabilities.argmax().item()
    return labels[predicted_class_id]

# Apply sentiment analysis to the 'content' column
df_reviews['sentiment'] = df_reviews['content'].apply(get_sentiment)

In [ ]:
print("Reviews with sentiment analysis:")
display(df_reviews.head())

# Also display sentiment distribution
print("\nSentiment Distribution:")
display(df_reviews['sentiment'].value_counts())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.countplot(x='sentiment', data=df_reviews, palette='viridis', hue='sentiment', legend=False, order=df_reviews['sentiment'].value_counts().index)
plt.title('Distribution of Sentiments in Google Play Reviews')
plt.xlabel('Sentiment')
plt.ylabel('Number of Reviews')
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()